# Step 9 — Prepare inputs for the API-based LLM evaluation

Builds the anchor set and candidate pairs for the larger, API-judged evaluation (DeepSeek V4 Pro + Mistral Large 3) that will become the headline evaluation, replacing the 12-anchor chat run. **This notebook prepares inputs only.**

**Hard constraints honoured here**
- No API calls, no API keys anywhere.
- Reads existing artifacts; writes ONLY under a new directory `evaluation_api/`. It never modifies or overwrites any existing output file.
- Retrieval uses the exact function/logic from `06_fullscale_prediction.ipynb` Step 2 (thesis Section 3.6): unit-normalise, cosine = dot, **mask the query index to -inf before sorting**, take the top-k. Reused verbatim below.

**Located input paths** (verified present before authoring)

| Input | Path |
|---|---|
| Six trained models | `outputs/full_scale/skipgram/{pct_dropoff,kneedle}/{drop_*,sensitivity_*}/` (`embeddings.npy` + `product_id_to_index.json`) |
| Product / aisle / department | `outputs/step1/products_text.csv` (joined; has `product_name`, `aisle`, `department`) |
| Basket counts (precomputed) | `outputs/full_scale/product_basket_freq.json` |
| Prior order-product table (fallback) | `data/order_products__prior.csv` |
| Original chat anchors (12) | `outputs/full_scale/evaluation/anchors.json` |
| Original chat retrieval (key) | `outputs/full_scale/evaluation/evaluation_key.json` |
| Original chat judge-facing inputs | `outputs/full_scale/evaluation/pairs_{method}_{value}.md` |
| Chat verdicts | `outputs/substitute_verdicts (claude).md`, `outputs/substitute_verdicts (gemini).md` |

**Two things flagged (see the config cell and Part 1):**
1. Stratum boundary at exactly `n = 1000`. Figure 5.2 (notebooks 07/08) uses `(bc >= lo) & (bc < hi)` with buckets `(…,50), (50,1000), (1000,inf)`, so `n = 1000` falls in **popular**, not medium. That contradicts this task's prose ("medium 50 ≤ n ≤ 1000; popular n > 1000"). We **match Figure 5.2**.
2. `product_basket_freq.json`'s `total_baskets` = 3,214,874 = the number of distinct prior orders, so its `product_basket_count` already equals "number of prior orders containing the product." We use it directly (the CSV path is kept as a documented fallback, not read).

## 0. Paths, configuration, and the six models

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
OUT = ROOT / "outputs"
SKIPGRAM_DIR = OUT / "full_scale" / "skipgram"
CHAT_EVAL_DIR = OUT / "full_scale" / "evaluation"
FREQ_PATH = OUT / "full_scale" / "product_basket_freq.json"
PRIOR_CSV = DATA_DIR / "order_products__prior.csv"   # fallback only; not read (see note)

VERDICT_FILES = {
    "A_claude": OUT / "substitute_verdicts (claude).md",   # Evaluator A: Claude Sonnet 5
    "B_gemini": OUT / "substitute_verdicts (gemini).md",   # Evaluator B: Gemini 3.1 Pro
}

# ---- All NEW outputs go here; nothing existing is touched ----
API_DIR = ROOT / "evaluation_api"
API_DIR.mkdir(parents=True, exist_ok=True)

def _products_text_path():
    for p in (OUT / "products_text.csv", OUT / "step1" / "products_text.csv"):
        if p.exists():
            return p
    raise FileNotFoundError("products_text.csv not found in outputs/ or outputs/step1/")

PRODUCTS_TEXT = _products_text_path()

# ---- The six configurations. `label` matches evaluation_key.json config keys;
# `short` matches the verdict files' "Model" column and the pairs_*.md filenames. ----
CONFIGS = [
    {"method": "pct_dropoff", "value": 0.03, "value_folder": "drop_0.03",
     "label": "pct_dropoff (0.03)", "short": "pct_dropoff_0.03", "display": "drop-off delta=0.03"},
    {"method": "pct_dropoff", "value": 0.05, "value_folder": "drop_0.05",
     "label": "pct_dropoff (0.05)", "short": "pct_dropoff_0.05", "display": "drop-off delta=0.05"},
    {"method": "pct_dropoff", "value": 0.10, "value_folder": "drop_0.10",
     "label": "pct_dropoff (0.10)", "short": "pct_dropoff_0.10", "display": "drop-off delta=0.10"},
    {"method": "kneedle", "value": 0.3, "value_folder": "sensitivity_0.3",
     "label": "kneedle (0.3)", "short": "kneedle_0.3", "display": "Kneedle S=0.3"},
    {"method": "kneedle", "value": 0.5, "value_folder": "sensitivity_0.5",
     "label": "kneedle (0.5)", "short": "kneedle_0.5", "display": "Kneedle S=0.5"},
    {"method": "kneedle", "value": 1.0, "value_folder": "sensitivity_1.0",
     "label": "kneedle (1.0)", "short": "kneedle_1.0", "display": "Kneedle S=1.0"},
]
CONFIG_SHORTS = [c["short"] for c in CONFIGS]
ELIGIBILITY_CONFIG = "kneedle_1.0"    # eligible pool = products in Kneedle S=1.0 vocabulary

# ---- Sampling / strata parameters ----
SAMPLE_SEED = 2026            # dedicated seed for this API selection (recorded in outputs)
PER_CELL_N = 10               # anchors per (department, stratum) cell
EXCLUDE_DEPARTMENT = "missing"  # 20 departments = all 21 except 'missing'

# Density strata — MATCHING Figure 5.2 (notebooks 07/08): (bc >= lo) & (bc < hi).
# NOTE: this puts exactly bc == 1000 in "popular" (dense), which differs from the
# task prose (medium 50<=n<=1000). We match the notebook, as instructed.
def stratum_of(bc):
    if bc < 50:
        return "sparse"      # thin  : bc < 50
    if bc < 1000:
        return "medium"      # medium: 50 <= bc < 1000
    return "popular"         # dense : bc >= 1000   (n == 1000 -> popular, per Figure 5.2)
STRATA = ["sparse", "medium", "popular"]

print("Output dir      :", API_DIR)
print("products_text   :", PRODUCTS_TEXT)
print("eligibility vocab:", ELIGIBILITY_CONFIG)
print("sample seed     :", SAMPLE_SEED)
print("strata rule     : sparse bc<50 | medium 50<=bc<1000 | popular bc>=1000  (matches Figure 5.2)")

Output dir      : /Users/linnhtet/Developer/bachelor-thesis/evaluation_api
products_text   : /Users/linnhtet/Developer/bachelor-thesis/outputs/step1/products_text.csv
eligibility vocab: kneedle_1.0
sample seed     : 2026
strata rule     : sparse bc<50 | medium 50<=bc<1000 | popular bc>=1000  (matches Figure 5.2)


## Retrieval — reused verbatim from `06_fullscale_prediction.ipynb` Step 2 (Section 3.6)

`load_model_vectors` and the sims/mask/argsort block are copied unchanged from the original evaluation notebook. `retrieve_topk` factors the identical per-anchor block (`sims = emb @ emb[a_idx]; sims[a_idx] = -np.inf; argsort` desc, take k) into a function so the same code drives both the top-5 main set and the top-10 bridge set.

In [2]:
def load_model_vectors(cfg):
    """Unit-normalised embedding matrix + id maps for one config (numpy pair only).
    Verbatim from 06_fullscale_prediction.ipynb Step 2."""
    d = SKIPGRAM_DIR / cfg["method"] / cfg["value_folder"]
    emb = np.load(d / "embeddings.npy").astype(np.float32)
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)
    with open(d / "product_id_to_index.json") as f:
        pid_to_idx = {int(pid): i for pid, i in json.load(f).items()}
    idx_to_pid = {i: pid for pid, i in pid_to_idx.items()}
    return emb, pid_to_idx, idx_to_pid


def retrieve_topk(model, anchor_pid, k):
    """Top-k neighbours of anchor_pid as [(candidate_pid, cosine), ...], using the
    EXACT logic of the original Step 2: mask the query index to -inf BEFORE sorting.
    Returns None if the anchor is not in this model's vocabulary."""
    emb, pid_to_idx, idx_to_pid = model
    if anchor_pid not in pid_to_idx:
        return None
    a_idx = pid_to_idx[anchor_pid]
    sims = emb @ emb[a_idx]                       # cosine (unit-norm) vs every product
    sims[a_idx] = -np.inf                         # mask self BEFORE sorting
    order = np.argsort(-sims)[:k]                 # top-k indices, descending
    return [(int(idx_to_pid[int(i)]), float(sims[int(i)])) for i in order]


# ---- Load all six models once; reuse across Part 2 (top-5) and Part 3 (top-10) ----
MODELS = {}
for c in CONFIGS:
    MODELS[c["short"]] = load_model_vectors(c)
    print(f"loaded {c['short']:<18} vocab={len(MODELS[c['short']][1]):,}")

loaded pct_dropoff_0.03   vocab=32,295
loaded pct_dropoff_0.05   vocab=39,104
loaded pct_dropoff_0.10   vocab=43,851
loaded kneedle_0.3        vocab=44,765
loaded kneedle_0.5        vocab=45,372
loaded kneedle_1.0        vocab=48,887


## Load tables, basket counts, vocabularies, and the original 12 anchors

In [3]:
# ---- Product metadata (name / aisle / department) ----
ptext = pd.read_csv(PRODUCTS_TEXT)
id_to_name = dict(zip(ptext["product_id"], ptext["product_name"]))
id_to_aisle = dict(zip(ptext["product_id"], ptext["aisle"]))
id_to_dept = dict(zip(ptext["product_id"], ptext["department"]))

def name(pid):  return id_to_name.get(int(pid), f"?{pid}")
def aisle(pid): return id_to_aisle.get(int(pid), "?")
def dept(pid):  return id_to_dept.get(int(pid), "?")

# ---- Basket counts = prior orders containing the product (precomputed) ----
with open(FREQ_PATH) as f:
    _freq = json.load(f)
basket_count = {int(p): int(c) for p, c in _freq["product_basket_count"].items()}
N_PRIOR_ORDERS = _freq["total_baskets"]
assert N_PRIOR_ORDERS == 3214874, "product_basket_freq total_baskets is not the prior order count"
def bc(pid): return basket_count.get(int(pid), 0)

# ---- Per-config vocabulary sets (pid membership) ----
VOCAB = {short: set(model[1].keys()) for short, model in MODELS.items()}
eligible_vocab = VOCAB[ELIGIBILITY_CONFIG]     # eligible pool = Kneedle S=1.0 vocabulary

# ---- The 12 original chat anchors (excluded from the main random pool) ----
with open(CHAT_EVAL_DIR / "anchors.json") as f:
    _orig = json.load(f)["anchors"]
ORIGINAL_ANCHORS = [int(a["product_id"]) for a in _orig]
ORIGINAL_LABEL = {int(a["product_id"]): a["label"] for a in _orig}   # pid -> A01..A12
print(f"products_text rows     : {len(ptext):,}")
print(f"eligible vocab (Kneedle 1.0): {len(eligible_vocab):,} products")
print(f"original chat anchors  : {len(ORIGINAL_ANCHORS)} (excluded from the main pool)")
print(f"departments (excl. '{EXCLUDE_DEPARTMENT}'): "
      f"{sorted(set(ptext['department'].dropna()) - {EXCLUDE_DEPARTMENT})}")

products_text rows     : 49,688
eligible vocab (Kneedle 1.0): 48,887 products
original chat anchors  : 12 (excluded from the main pool)
departments (excl. 'missing'): ['alcohol', 'babies', 'bakery', 'beverages', 'breakfast', 'bulk', 'canned goods', 'dairy eggs', 'deli', 'dry goods pasta', 'frozen', 'household', 'international', 'meat seafood', 'other', 'pantry', 'personal care', 'pets', 'produce', 'snacks']


## Part 1 — Stratified anchor sampling

Eligible pool = products in the **Kneedle S=1.0** vocabulary (not all six — that would let drop-off 0.03 decide which sparse products are eligible), excluding the 12 original chat anchors and the `missing` department. Each product is placed in a `(department, stratum)` cell; we draw `PER_CELL_N = 10` per cell with `np.random.default_rng(2026)` (cells visited in a fixed sorted order so the draw is reproducible). Cells with fewer than 10 eligible products contribute all of them (shortfall reported). Output: `evaluation_api/anchors.csv` with one boolean vocabulary-membership column per config.

In [5]:
DEPARTMENTS = sorted(d for d in set(ptext["department"].dropna()) if d != EXCLUDE_DEPARTMENT)
assert len(DEPARTMENTS) == 20, f"expected 20 departments, got {len(DEPARTMENTS)}"

# Build the eligible pool: (pid, dept, stratum, basket_count) for Kneedle-1.0 vocab
# products with a real (non-missing) department, excluding the 12 originals.
orig_set = set(ORIGINAL_ANCHORS)
pool = []
for pid in eligible_vocab:
    if pid in orig_set:
        continue
    d = id_to_dept.get(int(pid))
    if d is None or (isinstance(d, float)) or d == EXCLUDE_DEPARTMENT:
        continue
    if d not in DEPARTMENTS:
        continue
    pool.append((int(pid), d, stratum_of(bc(pid)), bc(pid)))

pool_df = pd.DataFrame(pool, columns=["product_id", "department", "stratum", "basket_count"])
print(f"eligible pool: {len(pool_df):,} products across "
      f"{pool_df['department'].nunique()} departments x {pool_df['stratum'].nunique()} strata")

# ---- Sample per (department, stratum) cell, deterministic order + one seeded rng ----
rng = np.random.default_rng(SAMPLE_SEED)
sampled_rows, cell_fill = [], []
for d in DEPARTMENTS:                       # fixed order -> reproducible draw
    for s in STRATA:
        cell = pool_df[(pool_df["department"] == d) & (pool_df["stratum"] == s)]
        ids = np.sort(cell["product_id"].to_numpy())     # canonical order before the draw
        n_elig = len(ids)
        if n_elig <= PER_CELL_N:
            chosen = ids                                  # take all (shortfall if < 10)
        else:
            chosen = rng.choice(ids, size=PER_CELL_N, replace=False)
        cell_fill.append({"department": d, "stratum": s, "eligible": int(n_elig),
                          "sampled": int(len(chosen)),
                          "shortfall": int(max(0, PER_CELL_N - n_elig))})
        for pid in chosen:
            sampled_rows.append({"product_id": int(pid), "department": d, "stratum": s})

cell_fill_df = pd.DataFrame(cell_fill)
short_cells = cell_fill_df[cell_fill_df["shortfall"] > 0]
print(f"sampled {len(sampled_rows)} anchors from {len(cell_fill_df)} cells "
      f"({(cell_fill_df['sampled'] == PER_CELL_N).sum()} full, "
      f"{len(short_cells)} short of {PER_CELL_N})")

# ---- Assemble anchors.csv with per-config vocabulary-membership booleans ----
anchor_records = []
for r in sampled_rows:
    pid = r["product_id"]
    rec = {"product_id": pid, "product_name": name(pid), "aisle": aisle(pid),
           "department": r["department"], "basket_count": bc(pid), "stratum": r["stratum"]}
    for c in CONFIGS:
        rec[f"in_{c['short']}"] = pid in VOCAB[c["short"]]
    anchor_records.append(rec)

anchors_df = pd.DataFrame(anchor_records).sort_values(
    ["department", "stratum", "product_id"]).reset_index(drop=True)
anchors_df.to_csv(API_DIR / "anchors.csv", index=False)
print(f"Saved {API_DIR.name}/anchors.csv  ({len(anchors_df)} anchors, "
      f"seed={SAMPLE_SEED})")
anchors_df.head(8)

eligible pool: 47,682 products across 20 departments x 3 strata
sampled 584 anchors from 60 cells (56 full, 4 short of 10)
Saved evaluation_api/anchors.csv  (584 anchors, seed=2026)


,product_id,product_name,aisle,department,basket_count,stratum,in_pct_dropoff_0.03,in_pct_dropoff_0.05,in_pct_dropoff_0.10,in_kneedle_0.3,in_kneedle_0.5,in_kneedle_1.0
0,5283,"Lager, Jamaican Style",beers coolers,alcohol,64,medium,True,True,True,True,True,True
1,6936,Day Time Fractional IPA,beers coolers,alcohol,71,medium,True,True,True,True,True,True
2,13745,Cinnamon Whiskey,spirits,alcohol,174,medium,False,True,True,True,True,True
3,14060,A Trois Silk Red,red wines,alcohol,54,medium,True,True,True,True,True,True
4,29610,Cordon Negro Brut,specialty wines champagnes,alcohol,192,medium,True,True,True,True,True,True
5,31142,Small Batch 9 Year Bourbon,spirits,alcohol,67,medium,False,True,True,True,True,True
6,36600,Beer Cans,beers coolers,alcohol,353,medium,True,True,True,True,True,True
7,38727,"Merlot, Columbia Valley, 2007",red wines,alcohol,82,medium,True,True,True,True,True,True


## Part 2 — Candidate retrieval (top-5) and deduplication

For every sampled anchor and every config whose vocabulary contains it, retrieve the top-5 nearest neighbours with the reused function. `retrievals.csv` is the long per-config table (cosine kept for our analysis only, never shown to judges). `pairs_unique.csv` deduplicates to one row per `(anchor_id, candidate_id)` across all configs, keeping exactly the fields the original protocol showed judges (anchor name; candidate name, aisle, department), plus the number of configs that retrieved it and a stable `pair_id`.

In [6]:
TOP_K_MAIN = 5
sampled_ids = anchors_df["product_id"].tolist()

retr_rows = []
for c in CONFIGS:
    short = c["short"]
    model = MODELS[short]
    for apid in sampled_ids:
        res = retrieve_topk(model, apid, TOP_K_MAIN)     # None if anchor not in this vocab
        if res is None:
            continue
        for rank, (cpid, score) in enumerate(res, start=1):
            retr_rows.append({"config": short, "anchor_id": apid, "rank": rank,
                              "candidate_id": cpid, "cosine_similarity": score})

retrievals_df = pd.DataFrame(retr_rows)
retrievals_df.to_csv(API_DIR / "retrievals.csv", index=False)
print(f"Saved {API_DIR.name}/retrievals.csv  ({len(retrievals_df):,} rows = "
      f"config x anchor-in-vocab x rank1..{TOP_K_MAIN})")

def build_pairs(df, id_prefix):
    """Dedup a long retrieval frame to unique (anchor_id, candidate_id) pairs with
    the judge-facing fields + n_configs + a stable pair_id (sorted for stability)."""
    g = (df.groupby(["anchor_id", "candidate_id"])["config"]
           .nunique().reset_index(name="n_configs_retrieved"))
    g = g.sort_values(["anchor_id", "candidate_id"]).reset_index(drop=True)
    g.insert(0, "pair_id", [f"{id_prefix}{i:05d}" for i in range(1, len(g) + 1)])
    g["anchor_name"] = g["anchor_id"].map(name)
    g["candidate_name"] = g["candidate_id"].map(name)
    g["candidate_aisle"] = g["candidate_id"].map(aisle)
    g["candidate_department"] = g["candidate_id"].map(dept)
    return g[["pair_id", "anchor_id", "anchor_name", "candidate_id", "candidate_name",
              "candidate_aisle", "candidate_department", "n_configs_retrieved"]]

pairs_unique_df = build_pairs(retrievals_df, "P")
pairs_unique_df.to_csv(API_DIR / "pairs_unique.csv", index=False)
n_retrieved_main = len(retrievals_df)
n_unique_main = len(pairs_unique_df)
print(f"Saved {API_DIR.name}/pairs_unique.csv  ({n_unique_main:,} unique pairs "
      f"from {n_retrieved_main:,} retrieved; "
      f"dedup rate {1 - n_unique_main / max(n_retrieved_main, 1):.1%})")
pairs_unique_df.head(6)

Saved evaluation_api/retrievals.csv  (15,795 rows = config x anchor-in-vocab x rank1..5)
Saved evaluation_api/pairs_unique.csv  (12,732 unique pairs from 15,795 retrieved; dedup rate 19.4%)


,pair_id,anchor_id,anchor_name,candidate_id,candidate_name,candidate_aisle,candidate_department,n_configs_retrieved
0,P00001,54,24/7 Performance Cat Litter,528,Tasty Treasures Cat Food Variety Pack,cat food care,pets,1
1,P00002,54,24/7 Performance Cat Litter,3074,Seafood Sensations Dry Cat Food,cat food care,pets,1
2,P00003,54,24/7 Performance Cat Litter,11780,Ultra Last Clumping Litter,cat food care,pets,2
3,P00004,54,24/7 Performance Cat Litter,14916,Purely Natural Treats Variety Pack Cat Treats,missing,missing,1
4,P00005,54,24/7 Performance Cat Litter,16542,Flutter Balls,cat food care,pets,1
5,P00006,54,24/7 Performance Cat Litter,16647,Double Duty Advanced Odor Control Clumping Cat...,cat food care,pets,1


## Part 3 — Bridge set (original 12 anchors, top-10) + match against the chat run

Retrieve the top-10 per config for the 12 original anchors (same k as the chat run), so API verdicts can be compared pair-by-pair with the chat verdicts. We check the freshly retrieved top-10 against `evaluation_key.json` (the exact candidates + order the chat run used); with the models unchanged this must match exactly, and any mismatch is reported. `pairs_bridge.csv` uses the same schema as `pairs_unique.csv`; overlaps with the main set are flagged (expected zero, since the 12 bridge anchors are excluded from the main pool).

In [7]:
TOP_K_BRIDGE = 10
with open(CHAT_EVAL_DIR / "evaluation_key.json") as f:
    KEY = json.load(f)

bridge_retr_rows, mismatches = [], []
for c in CONFIGS:
    short, lab = c["short"], c["label"]
    model = MODELS[short]
    key_anchors = KEY["configs"][lab]["anchors"]           # A01..A12
    # map anchor label -> product_id for this config's key block
    lbl_to_pid = {al: int(a["anchor_product_id"]) for al, a in key_anchors.items()}
    for apid in ORIGINAL_ANCHORS:
        res = retrieve_topk(model, apid, TOP_K_BRIDGE)
        if res is None:
            mismatches.append({"config": short, "anchor_id": apid,
                               "issue": "anchor not in this config's vocabulary"})
            continue
        fresh_ids = [cpid for cpid, _ in res]
        for rank, (cpid, score) in enumerate(res, start=1):
            bridge_retr_rows.append({"config": short, "anchor_id": apid, "rank": rank,
                                     "candidate_id": cpid, "cosine_similarity": score})
        # compare to the chat key's ordered candidate ids for this anchor
        al = ORIGINAL_LABEL.get(apid)
        key_block = key_anchors.get(al, {}).get("candidates", {})
        key_ids = [key_block[cl]["product_id"] for cl in sorted(key_block)]  # C01..C10 in rank order
        if key_ids != fresh_ids:
            mismatches.append({"config": short, "anchor_id": apid,
                               "issue": "top-10 differs from chat key",
                               "fresh": fresh_ids, "chat": key_ids})

bridge_retr_df = pd.DataFrame(bridge_retr_rows)
bridge_retr_df.to_csv(API_DIR / "retrievals_bridge.csv", index=False)

pairs_bridge_df = build_pairs(bridge_retr_df, "B")
# flag overlaps with the main unique set on (anchor_id, candidate_id)
main_pairset = set(zip(pairs_unique_df["anchor_id"], pairs_unique_df["candidate_id"]))
pairs_bridge_df["overlaps_main"] = [
    (a, cc) in main_pairset for a, cc in zip(pairs_bridge_df["anchor_id"], pairs_bridge_df["candidate_id"])]
pairs_bridge_df.to_csv(API_DIR / "pairs_bridge.csv", index=False)

n_retrieved_bridge = len(bridge_retr_df)
n_unique_bridge = len(pairs_bridge_df)
n_overlap = int(pairs_bridge_df["overlaps_main"].sum())
print(f"Bridge: {n_retrieved_bridge} retrieved (12 anchors x 6 configs x {TOP_K_BRIDGE}), "
      f"{n_unique_bridge} unique pairs")
print(f"overlap with main set: {n_overlap} "
      f"(expected 0 — bridge anchors are excluded from the main pool)")
if mismatches:
    print(f"\n!! {len(mismatches)} mismatch(es) vs the chat key:")
    for m in mismatches:
        print("   ", {k: v for k, v in m.items() if k in ('config', 'anchor_id', 'issue')})
else:
    print("Fresh top-10 matches the chat key exactly for all 12 anchors x 6 configs.")

Bridge: 720 retrieved (12 anchors x 6 configs x 10), 561 unique pairs
overlap with main set: 0 (expected 0 — bridge anchors are excluded from the main pool)
Fresh top-10 matches the chat key exactly for all 12 anchors x 6 configs.


## Part 4 — Chat verdict order check and Precision@5

We (1) confirm the chat *input* files (`pairs_*.md`) listed candidates in retrieval rank order, (2) check whether each verdict file preserved that order (verdict candidate-name sequence vs the key's rank-ordered names, per config x anchor), and (3) if order is preserved, compute **Precision@5** per config for both evaluators from the first 5 candidates per anchor. This table is diagnostic only — it is **not** written into the thesis. The 5 previously unmatched pairs (judge name-transcription differences) are identified by name and reported; because the order is preserved, position-based Precision@5 is unaffected by them.

In [8]:
def _norm(s):
    return re.sub(r"\s+", " ", str(s)).strip().casefold()

def read_md_verdicts(path):
    """Parse the '## Verdicts' markdown table -> ordered list of dicts
    {model, anchor, candidate, verdict} in file order (skips header/separator)."""
    rows, in_verdicts = [], False
    with open(path) as f:
        for line in f:
            s = line.rstrip("\n")
            if s.strip().startswith("## "):
                in_verdicts = (s.strip().lower() == "## verdicts")
                continue
            if not in_verdicts or not s.lstrip().startswith("|"):
                continue
            cellv = [x.strip() for x in s.strip().strip("|").split("|")]
            if len(cellv) != 4 or cellv[0].lower() == "model" or set(cellv[0]) <= set("-: "):
                continue
            if cellv[3].upper() not in ("YES", "NO"):
                continue
            rows.append({"model": cellv[0], "anchor": cellv[1],
                         "candidate": cellv[2], "verdict": cellv[3].upper()})
    return rows

# Chat inputs are in rank order by construction (06 wrote pairs_*.md rank-1 first);
# confirm the pairs_*.md candidate order equals the key's rank order for one config.
def key_rank_names(config_label, anchor_label):
    cb = KEY["configs"][config_label]["anchors"][anchor_label]["candidates"]
    return [cb[cl]["name"] for cl in sorted(cb)]   # C01..C10

label_by_short = {c["short"]: c["label"] for c in CONFIGS}
pid_by_name = {}      # (not needed) — we align by name within (config, anchor)
anchor_label_by_name = {a["anchor_name"]: al
                        for lab in [CONFIGS[0]["label"]]
                        for al, a in KEY["configs"][lab]["anchors"].items()}

verdicts = {ev: read_md_verdicts(p) for ev, p in VERDICT_FILES.items()}
for ev, rows in verdicts.items():
    print(f"{ev}: {len(rows)} verdict rows parsed")

# ---- order preservation + unmatched (transcription) pairs, per evaluator ----
def analyse_verdicts(rows):
    # group verdict rows by (model, anchor) preserving file order
    from collections import defaultdict, OrderedDict
    grp = OrderedDict()
    for r in rows:
        grp.setdefault((r["model"], r["anchor"]), []).append(r)
    order_ok, order_bad, unmatched = 0, 0, []
    prec5 = defaultdict(list)   # config short -> per-anchor precision@5
    for (model, anchor_name), vr in grp.items():
        lab = label_by_short.get(model)
        al = anchor_label_by_name.get(anchor_name)
        key_names = key_rank_names(lab, al) if (lab and al) else []
        vnames = [x["candidate"] for x in vr]
        # order preserved iff verdict names == key names position-by-position (by norm)
        same = len(vnames) == len(key_names) and all(
            _norm(a) == _norm(b) for a, b in zip(vnames, key_names))
        if same:
            order_ok += 1
        else:
            order_bad += 1
        # record name-transcription differences at matching positions
        for i, (a, b) in enumerate(zip(vnames, key_names)):
            if _norm(a) != _norm(b):
                unmatched.append({"model": model, "anchor": anchor_name, "rank": i + 1,
                                  "verdict_name": a, "retrieved_name": b})
        # Precision@5 from the first 5 verdict rows (file order = rank order when preserved)
        first5 = vr[:5]
        if first5:
            prec5[model].append(sum(x["verdict"] == "YES" for x in first5) / len(first5))
    return order_ok, order_bad, unmatched, prec5

print("\nOrder preservation + Precision@5 (diagnostic; NOT written to the thesis):")
p5_table = {}
for ev, rows in verdicts.items():
    ok, bad, unmatched, prec5 = analyse_verdicts(rows)
    print(f"\n[{ev}] (config,anchor) blocks with order preserved: {ok}, not preserved: {bad}")
    print(f"       name-transcription differences (unmatched pairs): {len(unmatched)}")
    for u in unmatched:
        print(f"         {u['model']} | {u['anchor']} | rank {u['rank']}: "
              f"verdict='{u['verdict_name']}' vs retrieved='{u['retrieved_name']}'")
    p5_table[ev] = {sh: (float(np.mean(prec5[sh])) if prec5.get(sh) else None) for sh in CONFIG_SHORTS}

print("\nPrecision@5 per config (mean over 12 anchors of YES-in-first-5 / 5):")
print(f"{'config':<18}" + "".join(f"{ev:>14}" for ev in verdicts))
for sh in CONFIG_SHORTS:
    print(f"{sh:<18}" + "".join(
        f"{(f'{p5_table[ev][sh]:.3f}' if p5_table[ev][sh] is not None else 'n/a'):>14}" for ev in verdicts))

A_claude: 720 verdict rows parsed
B_gemini: 720 verdict rows parsed

Order preservation + Precision@5 (diagnostic; NOT written to the thesis):

[A_claude] (config,anchor) blocks with order preserved: 67, not preserved: 5
       name-transcription differences (unmatched pairs): 5
         kneedle_0.5 | Flour Burrito Caseras Tortillas | rank 9: verdict='Whole Wheat Wraps 12"' vs retrieved='Whole Wheat Wraps 12\"'
         kneedle_1.0 | Flour Burrito Caseras Tortillas | rank 8: verdict='8" Gluten Free Soft Taco Tortillas' vs retrieved='8\" Gluten Free Soft Taco Tortillas'
         kneedle_1.0 | Plastic Cups | rank 3: verdict='Cut Crystal 10" Plastic Plates' vs retrieved='Cut Crystal 10\" Plastic Plates'
         pct_dropoff_0.05 | Flour Burrito Caseras Tortillas | rank 4: verdict='6" Yellow Corn Tortillas' vs retrieved='6\" Yellow Corn Tortillas'
         pct_dropoff_0.05 | Plastic Cups | rank 7: verdict='Large 10.5" Plates' vs retrieved='Large 10.5\" Plates'

[B_gemini] (config,anchor) b

## Part 5 — Selection report + cost projection

Writes `evaluation_api/selection_report.md` (cell fill, per-config coverage per stratum, dedup rates, cost projection) and `evaluation_api/selection_meta.json` (seed, strata rule, counts). The cost cell is parameterised with the given defaults; both judges score every pair, so cost is over the **unique** pair set (deduplicated), for (a) main only and (b) main + bridge.

In [9]:
# ---- cost model (parameterised; both judges score every unique pair) ----
TOK_IN, TOK_OUT = 850, 30
PRICES = {   # $ per 1M tokens: (input, output)
    "DeepSeek V4 Pro": (0.87, 1.74),
    "Mistral Large 3": (0.50, 1.50),
}
BUFFER = 0.30

def per_pair_cost(pin, pout):
    return (TOK_IN * pin + TOK_OUT * pout) / 1e6

def project(n_pairs):
    rows, total = {}, 0.0
    for model, (pin, pout) in PRICES.items():
        cst = n_pairs * per_pair_cost(pin, pout)
        rows[model] = cst
        total += cst
    return rows, total, total * (1 + BUFFER)

# unique pair counts: main, and main+bridge (union on (anchor_id, candidate_id))
union_pairs = main_pairset | set(zip(pairs_bridge_df["anchor_id"], pairs_bridge_df["candidate_id"]))
n_main = n_unique_main
n_main_plus_bridge = len(union_pairs)

# ---- per-config coverage of sampled anchors, per stratum ----
cov_lines = []
for c in CONFIGS:
    col = f"in_{c['short']}"
    for s in STRATA:
        sub = anchors_df[anchors_df["stratum"] == s]
        cov_lines.append({"config": c["short"], "stratum": s,
                          "covered": int(sub[col].sum()), "sampled": int(len(sub))})
cov_df = pd.DataFrame(cov_lines)

def md_table(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(out)

lines = []
lines.append("# API evaluation — selection report\n")
lines.append(f"- Sampling seed: **{SAMPLE_SEED}**")
lines.append(f"- Eligible pool: products in the **{ELIGIBILITY_CONFIG}** vocabulary "
             f"({len(eligible_vocab):,}), minus the 12 chat anchors, minus department "
             f"'{EXCLUDE_DEPARTMENT}'.")
lines.append(f"- Strata (matching Figure 5.2): sparse `bc<50`, medium `50<=bc<1000`, "
             f"popular `bc>=1000`. NOTE: exactly `bc==1000` -> popular (differs from the "
             f"task prose `50<=n<=1000`); matched to the notebook as instructed.")
lines.append(f"- Basket count = prior orders containing the product, from "
             f"`product_basket_freq.json` (total_baskets={N_PRIOR_ORDERS:,} = prior orders).")
lines.append(f"- Anchors sampled: **{len(anchors_df)}** ({PER_CELL_N}/cell x 20 depts x 3 strata "
             f"= {20 * 3 * PER_CELL_N} max).\n")

lines.append("## Cell fill (department x stratum)\n")
lines.append(md_table(cell_fill_df))
sc = cell_fill_df[cell_fill_df['shortfall'] > 0]
lines.append(f"\nCells short of {PER_CELL_N}: **{len(sc)}** "
             f"(total shortfall {int(cell_fill_df['shortfall'].sum())} anchors).\n")

lines.append("## Per-config coverage of sampled anchors, per stratum\n")
lines.append(md_table(cov_df))
lines.append("")

lines.append("## Pair counts and deduplication\n")
lines.append(f"- Main: retrieved {n_retrieved_main:,} (top-5), unique {n_unique_main:,}, "
             f"dedup rate {1 - n_unique_main / max(n_retrieved_main, 1):.1%}.")
lines.append(f"- Bridge: retrieved {n_retrieved_bridge:,} (top-10), unique {n_unique_bridge:,}, "
             f"dedup rate {1 - n_unique_bridge / max(n_retrieved_bridge, 1):.1%}.")
lines.append(f"- Bridge overlap with main: {n_overlap} (bridge anchors excluded from main).")
lines.append(f"- Main + bridge unique (union): **{n_main_plus_bridge:,}**.\n")

lines.append("## Cost projection\n")
lines.append(f"Assumptions: {TOK_IN} input + {TOK_OUT} output tokens per pair; both models "
             f"score every unique pair; {int(BUFFER*100)}% buffer.\n")
for scope, n in [("(a) main only", n_main), ("(b) main + bridge", n_main_plus_bridge)]:
    rows, total, buffered = project(n)
    lines.append(f"**{scope}** — {n:,} unique pairs")
    for model, cst in rows.items():
        lines.append(f"- {model}: ${cst:,.2f}")
    lines.append(f"- Total: ${total:,.2f}  |  Total +{int(BUFFER*100)}% buffer: **${buffered:,.2f}**\n")

lines.append("## Assumptions and flags\n")
lines.append("- Retrieval reused verbatim from `06_fullscale_prediction.ipynb` Step 2 "
             "(query masked to -inf before sorting; Section 3.6).")
lines.append("- `n==1000` boundary matched to Figure 5.2 (-> popular), not the task prose.")
lines.append("- Cost is over UNIQUE pairs (deduplicated); each pair judged once per model.")
lines.append("- Cosine similarities are kept in `retrievals.csv` for our analysis only and "
             "are never shown to the judges.")

report = "\n".join(lines) + "\n"
(API_DIR / "selection_report.md").write_text(report)

meta = {
    "sample_seed": SAMPLE_SEED, "per_cell_n": PER_CELL_N,
    "eligibility_config": ELIGIBILITY_CONFIG, "excluded_department": EXCLUDE_DEPARTMENT,
    "strata_rule": {"sparse": "bc<50", "medium": "50<=bc<1000", "popular": "bc>=1000",
                    "note": "matches Figure 5.2; bc==1000 -> popular"},
    "n_anchors": int(len(anchors_df)),
    "n_unique_pairs_main": int(n_main), "n_unique_pairs_bridge": int(n_unique_bridge),
    "n_unique_pairs_union": int(n_main_plus_bridge),
    "top_k_main": TOP_K_MAIN, "top_k_bridge": TOP_K_BRIDGE,
    "config_shorts": CONFIG_SHORTS,
}
(API_DIR / "selection_meta.json").write_text(json.dumps(meta, indent=2))
print("Wrote selection_report.md and selection_meta.json")
print(report)

Wrote selection_report.md and selection_meta.json
# API evaluation — selection report

- Sampling seed: **2026**
- Eligible pool: products in the **kneedle_1.0** vocabulary (48,887), minus the 12 chat anchors, minus department 'missing'.
- Strata (matching Figure 5.2): sparse `bc<50`, medium `50<=bc<1000`, popular `bc>=1000`. NOTE: exactly `bc==1000` -> popular (differs from the task prose `50<=n<=1000`); matched to the notebook as instructed.
- Basket count = prior orders containing the product, from `product_basket_freq.json` (total_baskets=3,214,874 = prior orders).
- Anchors sampled: **584** (10/cell x 20 depts x 3 strata = 600 max).

## Cell fill (department x stratum)

| department | stratum | eligible | sampled | shortfall |
|---|---|---|---|---|
| alcohol | sparse | 672 | 10 | 0 |
| alcohol | medium | 350 | 10 | 0 |
| alcohol | popular | 28 | 10 | 0 |
| babies | sparse | 439 | 10 | 0 |
| babies | medium | 510 | 10 | 0 |
| babies | popular | 125 | 10 | 0 |
| bakery | sparse | 46